<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/week2e.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

orders = pd.DataFrame({
    'order_id':   range(1001, 1011),
    'region':     ['Lagos','Abuja','Lagos','Kano','Abuja',
                   'Lagos','Ibadan','Kano','Abuja','Lagos'],
    'category':   ['Electronics','Home','Home','Electronics','Home',
                   'Electronics','Home','Home','Electronics','Home'],
    'units':      [10, 6, 9, 4, 13, 7, 8, 5, 12, 3],
    'unit_price': [200, 90, 65, 250, 100, 170, 60, 240, 120, 80],
})
orders['revenue'] = orders['units'] * orders['unit_price']

regions_meta = pd.DataFrame({
    'region':  ['Lagos', 'Abuja', 'Kano', 'Port Harcourt'],
    'manager': ['Ada', 'Femi', 'Zainab', 'Chidi'],
})

## Group by Region
I want to see the region perfomrmace by revenue

In [ ]:
# Group by region and sum the revenue column
region_revenue = orders.groupby('region')['revenue'].sum().reset_index()
region_revenue


,region,revenue
0,Abuja,3280
1,Ibadan,480
2,Kano,2200
3,Lagos,4015


## Group by Category and sum total units sold
I want to see the category performance and total units sold in each categories

In [5]:
# Group by category and find the sum of the units column
category_units = orders.groupby('category')['units'].sum().reset_index()
category_units


,category,units
0,Electronics,33
1,Home,44


## Aggregation
Total revenue per region in descending order with number of orders for each region.

In [ ]:
# Use .agg() with named aggregation to create clean, specific column names
region_summary = orders.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    average_revenue=('revenue', 'mean'),
    number_of_orders=('order_id', 'count')
).sort_values(by='total_revenue', ascending=False).reset_index()

region_summary


,region,total_revenue,average_revenue,number_of_orders
0,Lagos,4015,1003.750000,4
1,Abuja,3280,1093.333333,3
2,Kano,2200,1100.000000,2
3,Ibadan,480,480.000000,1


In [ ]:
# Filter the orders dataframe first, then group and sum
filtered_region_revenue = (
    orders[orders['units'] > 5]
    .groupby('region')['revenue']
    .sum()
    .reset_index()
)
filtered_region_revenue


,region,revenue
0,Abuja,3280
1,Ibadan,480
2,Lagos,3775


In [ ]:
# Group by multiple columns and use reset_index() to flatten the output
multi_group = (
    orders.groupby(['region', 'category'])['revenue']
    .sum()
    .reset_index()
)
multi_group


,region,category,revenue
0,Abuja,Electronics,1440
1,Abuja,Home,1840
2,Ibadan,Home,480
3,Kano,Electronics,1000
4,Kano,Home,1200
5,Lagos,Electronics,3190
6,Lagos,Home,825


In [ ]:
# Check shape before merge
print("Shape of orders before:", orders.shape)

# Perform left merge
left_merged = orders.merge(regions_meta, on='region', how='left')

# Check shape after merge
print("Shape after left merge:", left_merged.shape)
left_merged


Shape of orders before: (10, 6)
Shape after left merge: (10, 7)


,order_id,region,category,units,unit_price,revenue,manager
0,1001,Lagos,Electronics,10,200,2000,Ada
1,1002,Abuja,Home,6,90,540,Femi
2,1003,Lagos,Home,9,65,585,Ada
3,1004,Kano,Electronics,4,250,1000,Zainab
4,1005,Abuja,Home,13,100,1300,Femi
5,1006,Lagos,Electronics,7,170,1190,Ada
6,1007,Ibadan,Home,8,60,480,NaN
7,1008,Kano,Home,5,240,1200,Zainab
8,1009,Abuja,Electronics,12,120,1440,Femi
9,1010,Lagos,Home,3,80,240,Ada


In [ ]:
# Perform inner merge
inner_merged = orders.merge(regions_meta, on='region', how='inner')

print("Shape of orders before:", orders.shape)
print("Shape after inner merge:", inner_merged.shape)

# Code to find which orders disappeared:
missing_from_inner = orders[~orders['order_id'].isin(inner_merged['order_id'])]
print("\n--- Disappeared Orders ---")
missing_from_inner


Shape of orders before: (10, 6)
Shape after inner merge: (9, 7)

--- Disappeared Orders ---


,order_id,region,category,units,unit_price,revenue
6,1007,Ibadan,Home,8,60,480


In [ ]:
# Perform outer merge
outer_merged = orders.merge(regions_meta, on='region', how='outer')

print("Shape after outer merge:", outer_merged.shape)

# Find the row that has no matching order (where order_id is missing)
unmatched_meta_rows = outer_merged[outer_merged['order_id'].isna()]
print("\n--- Rows with no matching order ---")
unmatched_meta_rows


Shape after outer merge: (11, 7)

--- Rows with no matching order ---


,order_id,region,category,units,unit_price,revenue,manager
10,NaN,Port Harcourt,NaN,NaN,NaN,NaN,Chidi


In [ ]:
# This finds the 1 order that was kicked out of the inner merge
disappeared_order = orders[~orders['region'].isin(regions_meta['region'])]
print("The order that disappeared:")
disappeared_order


The order that disappeared:


,order_id,region,category,units,unit_price,revenue
6,1007,Ibadan,Home,8,60,480


In [ ]:
# This finds the 1 region from meta that has no matching orders
ghost_region = regions_meta[~regions_meta['region'].isin(orders['region'])]
print("The region with no matching orders:")
ghost_region


The region with no matching orders:


,region,manager
3,Port Harcourt,Chidi


In [ ]:
# 1. Merge the tables together using an inner merge
merged_df = orders.merge(regions_meta, on='region', how='inner')

# 2. Group by manager, sum the revenue, sort it descending, and clean the index
manager_revenue = (
    merged_df.groupby('manager')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# Display the final flat table
manager_revenue


,manager,revenue
0,Ada,4015
1,Femi,3280
2,Zainab,2200
